# Pricer with Qwen 2.5 3B

**Same task, different base model:** Fine-tune **Qwen 2.5 3B** on the instructor's price-prediction data (QLoRA), then evaluate and compare to Llama 3.2 3B results.

In [ ]:
!pip install -q --upgrade bitsandbytes trl transformers datasets peft accelerate
# For Colab: download course evaluator
!wget -q https://raw.githubusercontent.com/ed-donner/llm_engineering/main/week7/util.py -O util.py 2>/dev/null || true

In [ ]:
import os
import re
from datetime import datetime
from huggingface_hub import login
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from datasets import load_dataset
from peft import LoraConfig, PeftModel
from trl import SFTTrainer, SFTConfig
from google.colab import userdata


hf_token = userdata.get('HF_TOKEN')

login(token=hf_token, add_to_git_credential=True)

## Config: Qwen 2.5 3B + same task/data as Ed

In [ ]:
BASE_MODEL = "Qwen/Qwen2.5-3B"
PROJECT_NAME = "price"
HF_USER = "ed-donner"

DATA_USER = "ed-donner"
DATASET_NAME = f"{DATA_USER}/items_prompts_lite"

RUN_NAME = f"{datetime.now():%Y-%m-%d_%H.%M.%S}-qwen25-3b-lite"
PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
HUB_MODEL_NAME = f"{HF_USER}/{PROJECT_RUN_NAME}"

EPOCHS = 1
BATCH_SIZE = 32
MAX_SEQUENCE_LENGTH = 128
GRADIENT_ACCUMULATION_STEPS = 1

LORA_R = 32
LORA_ALPHA = LORA_R * 2
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
LORA_DROPOUT = 0.1

LEARNING_RATE = 1e-4
WARMUP_RATIO = 0.01
LR_SCHEDULER_TYPE = "cosine"
WEIGHT_DECAY = 0.001
OPTIMIZER = "paged_adamw_32bit"

VAL_SIZE = 500
TRAIN_SIZE = 2000
LOG_STEPS = 5
SAVE_STEPS = 100
LOG_TO_WANDB = False

## Load dataset and add \"text\" for SFT

In [ ]:
dataset = load_dataset(DATASET_NAME)
train_raw = dataset["train"].select(range(min(TRAIN_SIZE, len(dataset["train"]))))
train = train_raw.map(
    lambda r: {"text": r["prompt"] + r["completion"]},
    remove_columns=["prompt", "completion"],
)
val = dataset["val"].select(range(min(VAL_SIZE, len(dataset["val"])))).map(
    lambda r: {"text": r["prompt"] + r["completion"]},
    remove_columns=["prompt", "completion"],
)
test = dataset["test"]

print(f"Train: {len(train)}, Val: {len(val)}, Test: {len(test)}")

## QLoRA: load base model (4-bit) and tokenizer

In [ ]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)
base_model.generation_config.pad_token_id = tokenizer.pad_token_id
print(f"Memory footprint: {base_model.get_memory_footprint() / 1e6:.1f} MB")

## LoRA config and SFT training args

In [ ]:
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,
)

train_args = SFTConfig(
    output_dir=PROJECT_RUN_NAME,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    optim=OPTIMIZER,
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    logging_steps=LOG_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    max_grad_norm=0.3,
    gradient_checkpointing=False,
    warmup_steps=WARMUP_RATIO,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    report_to="wandb" if LOG_TO_WANDB else "none",
    run_name=RUN_NAME,
    max_length=MAX_SEQUENCE_LENGTH,
    save_strategy="steps",
    eval_strategy="steps",
    eval_steps=SAVE_STEPS,
)

## SFTTrainer and train

In [ ]:
trainer = SFTTrainer(
    model=base_model,
    args=train_args,
    train_dataset=train,
    eval_dataset=val,
    peft_config=lora_config,
)

trainer.train()

In [ ]:
trainer.save_model(PROJECT_RUN_NAME)
# trainer.push_to_hub()  # uncomment to push to HF (set HF_USER above)
print(f"Saved to {PROJECT_RUN_NAME}")

## Evaluate on test set (course evaluator)

Load the fine-tuned adapter and run the same evaluator as Week 7. Requires `util.py` in the same directory (or from repo).

In [ ]:
from util import evaluate

# Load base model + adapter (use same quant config as training)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)
model = PeftModel.from_pretrained(model, PROJECT_RUN_NAME)
model.eval()

def predictor(datapoint):
    prompt = datapoint["prompt"]
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=16, do_sample=False, pad_token_id=tokenizer.pad_token_id)
    text = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    return text

EVAL_SIZE = min(200, len(test))
test_list = [test[i] for i in range(EVAL_SIZE)]
evaluate(predictor, test_list, size=EVAL_SIZE)